<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.1-spm-and-thermal/Ex10.1_00_reference.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_10.1 · Notebook 00 — the reference solutions

**Paired with L10.1 · Battery models**

**Read and run; you are not asked to rewrite this.**

Two references, used for different purposes:

- **An analytic solution** for diffusion in a sphere with constant surface
  flux. Needs nothing installed, and lets you verify your residual before
  trusting anything else.
- **PyBaMM**, for the real LG M50 21700 cell — a public, published parameter
  set for an actual cylindrical cell.

Install PyBaMM with `!pip install -q pybamm`. Everything in this notebook
except section 3 works without it.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.1-spm-and-thermal/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
print("PyBaMM available:", pb.PYBAMM_OK)
if not pb.PYBAMM_OK:
    print("  install it with:   !pip install -q pybamm")

## 1 · The analytic particle solution

$$\frac{\partial c}{\partial t} = \frac{1}{r^2}
\frac{\partial}{\partial r}\left(r^2 \frac{\partial c}{\partial r}\right),
\qquad \left.\frac{\partial c}{\partial r}\right|_{0}=0, \qquad
\left.\frac{\partial c}{\partial r}\right|_{1}=1, \qquad c(r,0)=0$$

This is L8.2's heat equation in spherical coordinates. Verify it, then use it.

In [ ]:
rg = np.linspace(0.001, 1, 200)
plt.figure(figsize=(7, 3.4))
for t in (0.02, 0.05, 0.1, 0.3, 0.6, 1.0):
    plt.plot(rg, pb.analytic_sphere(rg, t), label=f"t = {t}")
plt.xlabel("r / R$_s$"); plt.ylabel("dimensionless concentration")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

### Sampling the (r, t) slab

`pb.particle_points` returns a plain NumPy array, like every sampler in
`pinn_core`: column 0 is $r$, column 1 is $t$. Nothing becomes a tensor until
`to_tensor` says so, and `requires_grad=True` is what makes a point
differentiable.

In [ ]:
# the points the PINN residual will be formed on
rt_np = pb.particle_points(400, t_end=1.0)
rt = to_tensor(rt_np, requires_grad=True)
r, t = rt[:, 0:1], rt[:, 1:2]            # column 0 is r, column 1 is t

c = to_tensor(pb.analytic_sphere(rt_np[:, 0], rt_np[:, 1]).reshape(-1, 1))
check_shape("collocation points", rt_np, (400, 2))
check_shape("analytic values", to_numpy(c), (400, 1))
print("  r spans", f"{rt_np[:, 0].min():.4f} .. {rt_np[:, 0].max():.4f}",
      "  t spans", f"{rt_np[:, 1].min():.4f} .. {rt_np[:, 1].max():.4f}")

# finite-difference check of the PDE (autograd needs a differentiable c)
h = 1e-4; r0, t0 = 0.53, 0.05
ct  = (pb.analytic_sphere(r0, t0 + h) - pb.analytic_sphere(r0, t0 - h)) / (2 * h)
d1  = (pb.analytic_sphere(r0 + h, t0) - pb.analytic_sphere(r0 - h, t0)) / (2 * h)
d2_ = (pb.analytic_sphere(r0 + h, t0) - 2 * pb.analytic_sphere(r0, t0)
       + pb.analytic_sphere(r0 - h, t0)) / h ** 2
print(f"PDE residual  = {abs(ct - (d2_ + 2 / r0 * d1))[0]:.2e}")
print(f"surface flux  = {((pb.analytic_sphere(1.0,t0)-pb.analytic_sphere(1.0-h,t0))/h)[0]:.5f}  (expect 1)")

## 2 · The scales you are up against

This is why L10.1 slide 16 insists on non-dimensionalisation.

In [ ]:
cell = pb.CellParams(c_rate=1.0)
print(cell)
print(f"  current              {cell.current:.1f} A")
print(f"  discharge duration   {cell.t_discharge:.0f} s")
print(f"  particle radius      {cell.Rs_n:.2e} m")
print(f"  solid diffusivity    {cell.Ds_n:.2e} m2/s")
print(f"  particle time const  {cell.tau_diff_n:.0f} s")
print(f"  c_s,max              {cell.cs_max_n:.0f} mol/m3")
print(f"  Biot number          {cell.biot:.4f}")
print("\n  ratio of largest to smallest raw quantity: "
      f"{cell.cs_max_n / cell.Ds_n:.1e}")

Note the particle time constant against the discharge duration. Their ratio is
what decides whether the SPM is adequate — and it changes with C-rate.

## 3 · The PyBaMM reference (needs PyBaMM)

In [ ]:
if pb.PYBAMM_OK:
    for rate in (0.5, 1.0, 2.0):
        ref = pb.pybamm_reference("SPMe", c_rate=rate, parameter_set="Chen2020")
        plt.plot(ref["t"] / 60, ref["V"], label=f"{rate}C")
    plt.xlabel("time [min]"); plt.ylabel("terminal voltage [V]")
    plt.legend(); plt.tight_layout(); plt.show()
else:
    print("PyBaMM not installed - run:  !pip install -q pybamm")

**A note on parameter sets.** `Chen2020` is complete for isothermal work but
carries no thermal parameters. For notebook 03 use `ORegan2022`, which measured
them — including the entropic coefficient ∂U/∂T that Gu & Wang had to neglect.
`pb.pybamm_reference` raises an error if you ask for thermal with Chen2020.

---

## 4 · Ready

Next: **notebook 01**, where the particle problem is solved for the first time
and the analytic solution above becomes the thing you are scored against.